In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import data_path
from src.spark_utils import get_spark
import pyspark.sql.functions as F
from src.datasets import read_qqp, read_qqp_benchmark, read_feature_benchmark
from src.near_duplicates import near_duplicates_pairs, near_duplicates_benchmark, compute_join_schema, optimal_design
from src.metrics import ModelMetrics, extract_metrics
from src.fingerprint import FingerprintParams, create_fingerprints, FINGERPRINT_SIZES

%load_ext autoreload
%autoreload 2

In [ ]:
print(compute_join_schema(f=64, d=20, K=6, p_min=2))
print(optimal_design(f=64, k=6, d=20, p_min=2, nibble=False))

## Feature Model Analysis

This section aims to compare the following feature models: bag_of_words, embeddings, shingles and tf_idf.
Models will compete on two datasets: feature_benchmark_easy and feature_benchmark_hard.
Each row of the dataset is flagged as "is_near_duplicate_semantic" and/or "is_near_duplicate_syntax".
The "easy" dataset contains 100 pairs of short sentences, while the "hard" one is made of 40 journal articles pairs (~250 words).


### bag_of_words

The simplest feature model. Two elements are equal iff they contain similar words, in any order.


In [ ]:
K = 6

test_df = df_wiki.sample(0.8)

test_df = create_fingerprints(test_df, model="tf-idf", input_col="data1", output_col="fingerprint1", fingerprint_size=64, other=None)
test_df = create_fingerprints(test_df, model="tf-idf", input_col="data2", output_col="fingerprint2", fingerprint_size=64, other=None)
test_df = near_duplicates_pairs(test_df, f1="fingerprint1", f2="fingerprint2", K=K)
test_df.show()

In [ ]:
fingerprint_size = 64
K = 17
shingle_size = 6

spark.catalog.clearCache()

fast_df = create_fingerprints(df_easy, method="embeddings", input_col="data1", output_col="fingerprint1", size=fingerprint_size, fast=True, other=FingerprintParams(shingle_size=shingle_size, over_words=False))
fast_df = create_fingerprints(fast_df, method="embeddings", input_col="data2", output_col="fingerprint2", size=fingerprint_size, fast=True, other=FingerprintParams(shingle_size=shingle_size, over_words=False))
metrics = near_duplicates_benchmark(fast_df, "fingerprint1", "fingerprint2", "is_near_duplicate_semantic", K=K)
fast_df.explain()

In [ ]:
fingerprint_size = 64
K = 6

fast_df = create_fingerprints(qqp, method="shingles", input_col="data1", output_col="fingerprint1", size=fingerprint_size)
fast_df = create_fingerprints(qqp, method="shingles", input_col="data2", output_col="fingerprint2", size=fingerprint_size)
fast_df.write.format("noop").mode("overwrite").save()

In [ ]:
%%time
fingerprint_size = 64
K = 17
shingle_size = 6

spark.catalog.clearCache()

slow_df = create_fingerprints(qqp, method="shingles", input_col="data1", output_col="fingerprint1", size=fingerprint_size, fast=False, other=FingerprintParams(shingle_size=shingle_size, over_words=False))
slow_df = create_fingerprints(qqp, method="shingles", input_col="data2", output_col="fingerprint2", size=fingerprint_size, fast=False, other=FingerprintParams(shingle_size=shingle_size, over_words=False))
slow_df.write.format("noop").mode("overwrite").save()

In [ ]:
import numpy as np
from src.simhash import _numba_fastsimhash

# Create 10 empty dummy hashes
dummy_matrix = np.zeros((10, 64), dtype=np.uint8)

_numba_fastsimhash(dummy_matrix, 10, 64)

# Now Numba has compiled _numba_fastsimhash
llvm_dict = _numba_fastsimhash.inspect_llvm()

# for signature, llvm_ir in llvm_dict.items():
    # print(llvm_ir) # Look for <8 x i32>, it's an SIMD instruction 

In [ ]:
spark.stop()